In [ ]:
pip install kafka-python

In [1]:
import json
import time
import random
from utils import load_config
from datetime import datetime, timedelta

class Batch:
    def __init__(self, batch_id, recipe, canteen_id, start_time):
        self.batch_id = batch_id
        self.recipe = recipe
        self.canteen_id = canteen_id
        self.driver_id = None
        self.state = "START" 
        self.ready_time = start_time 
        self.current_station_id = None 
        self.initial_weight = round(random.uniform(10.0, 50.0), 2)
        self.current_weight = self.initial_weight
        self.truck_temp = None 
        self.trip_duration = 0 
        self.simulated_durations = (
            {
                "START": 0, 
                "PREPARING": random.randint(15, 20), 
                "COOKING": random.randint(25, 55), 
                "PACKING": random.randint(10, 25), 
                "LOADING": random.randint(10, 15), 
                "DELIVERY": random.randint(30, 60)
            }
        )

    def can_advance(self, current_sim_time):
        return current_sim_time >= self.ready_time

    def advance(self):
        transitions = (
            {
                "START": "PREPARING", 
                "PREPARING": "COOKING", 
                "COOKING": "PACKING", 
                "PACKING": "LOADING", 
                "LOADING": "DELIVERY", 
                "DELIVERY": "DELIVERED"
            }
        )
        
        new_state = transitions.get(self.state, "DELIVERED")
        mins_to_add = self.simulated_durations.get(new_state, 0)
        
        if self.state == "COOKING" and new_state == "PACKING":
            self.current_weight = round(
                self.current_weight * random.uniform(0.97, 0.99), 2
            )
        
        if new_state == "LOADING" and self.truck_temp is None:
            if random.random() < 0.10:
                self.truck_temp = round(random.uniform(6.0, 15.0), 1)
            else:
                self.truck_temp = round(random.uniform(-18.0, 5.0), 1)
                
        if new_state == "DELIVERY":
            self.trip_duration = mins_to_add
                
        self.state = new_state
        return mins_to_add

class EventGenerator:
    def __init__(self, config_data):
        self.recipes = ["chicken_rice", "beef_stew", "veg_pasta", "fish_curry"]
        self.canteens = ["cant_01", "cant_02", "cant_03", "cant_04"]
        
        self.stations = {
            "prep_01": {"role": "PREPARING", "state": "IDLE", "batch_id": None},
            "prep_02": {"role": "PREPARING", "state": "IDLE", "batch_id": None},
            "prep_03": {"role": "PREPARING", "state": "IDLE", "batch_id": None},
            "prep_04": {"role": "PREPARING", "state": "IDLE", "batch_id": None},
            "cook_01": {"role": "COOKING", "state": "IDLE", "batch_id": None},
            "cook_02": {"role": "COOKING", "state": "IDLE", "batch_id": None},
            "cook_03": {"role": "COOKING", "state": "IDLE", "batch_id": None},
            "cook_04": {"role": "COOKING", "state": "IDLE", "batch_id": None},
            "cook_05": {"role": "COOKING", "state": "IDLE", "batch_id": None},
            "cook_06": {"role": "COOKING", "state": "IDLE", "batch_id": None},
            "pack_01": {"role": "PACKING", "state": "IDLE", "batch_id": None},
            "pack_02": {"role": "PACKING", "state": "IDLE", "batch_id": None},
            "pack_03": {"role": "PACKING", "state": "IDLE", "batch_id": None},
            "pack_04": {"role": "PACKING", "state": "IDLE", "batch_id": None}        
        }
        
        self.drivers_status = {
            "drv_01": {"state": "IDLE", "canteen_id": None, "active_batches": 0, "return_time": None},
            "drv_02": {"state": "IDLE", "canteen_id": None, "active_batches": 0, "return_time": None},
            "drv_03": {"state": "IDLE", "canteen_id": None, "active_batches": 0, "return_time": None},
            "drv_04": {"state": "IDLE", "canteen_id": None, "active_batches": 0, "return_time": None},
            "drv_05": {"state": "IDLE", "canteen_id": None, "active_batches": 0, "return_time": None},
            "drv_06": {"state": "IDLE", "canteen_id": None, "active_batches": 0, "return_time": None},
            "drv_07": {"state": "IDLE", "canteen_id": None, "active_batches": 0, "return_time": None}
        }
        
        self.recipe_ingredients = {
            "chicken_rice": {"chicken": 0.5, "rice": 0.4, "water": 0.1},
            "beef_stew": {"beef": 0.4, "potato": 0.4, "onion": 0.1, "beef_broth": 0.1},
            "veg_pasta": {"pasta": 0.3, "tomato_sauce": 0.5, "water": 0.2},
            "fish_curry": {"white_fish": 0.5, "curry_paste": 0.3, "coconut_milk": 0.2}
        }
        
        self.active_batches = {} 
        self.counter = 1000
        self.kitchen_event_counter = 1000
        self.dispatch_event_counter = 1000
        
        self.start_hour = config_data.get("daily_start_hour", 6)
        self.stop_hour = config_data.get("daily_stop_hour", 21)
        self.sim_days = config_data.get("simulation_days", 30)
        
        self.start_date = datetime(2026, 3, 15, self.start_hour, 0, 0)
        self.global_clock = self.start_date
        self.end_of_simulation = self.start_date + timedelta(days=self.sim_days - 1, hours=(self.stop_hour - self.start_hour))
        
        self.stop_new_batches_today = False
        self.total_rows_made = 0

        self.source = config_data.get("data_source_name", "data_generator")

    def get_ingredients(self, recipe, total_weight):
        items = self.recipe_ingredients.get(recipe, {})
        if total_weight is None:
            return [{"item": name, "amount_kg": None} for name in items.keys()]
        return [{"item": name, "amount_kg": round(total_weight * ratio, 2)} 
                for name, ratio in items.items()]

    def update_clock(self):
        self.global_clock += timedelta(minutes=1)
        for d_id, status in self.drivers_status.items():
            if status["state"] == "RETURNING" and status["return_time"] <= self.global_clock:
                status["state"] = "IDLE"
                status["return_time"] = None
                status["canteen_id"] = None

    def check_day_limits(self):
        if self.global_clock.hour >= self.stop_hour:
            self.stop_new_batches_today = True

        all_drivers_idle = all(status["state"] == "IDLE" for status in self.drivers_status.values())
        
        if self.stop_new_batches_today and not self.active_batches and all_drivers_idle:
            if self.global_clock >= self.end_of_simulation:
                return "DONE"
            
            next_day = self.global_clock + timedelta(days=1)
            self.global_clock = next_day.replace(hour=self.start_hour, minute=0, second=0)
            self.stop_new_batches_today = False
            print(f"\n--- Day ended. Jumping to next morning: {self.global_clock.date()} ---\n")
        return "CONTINUE"

    def spawn_new_batches(self):
        if not self.stop_new_batches_today and ((random.random() < 0.25 and len(self.active_batches) < 8) or not self.active_batches):
            self.counter += 1
            b_id = f"BATCH-{self.counter}"
            self.active_batches[b_id] = Batch(b_id, random.choice(self.recipes), random.choice(self.canteens), self.global_clock)

    def get_target_state(self, current_state):
        next_states = {
            "START": "PREPARING", 
            "PREPARING": "COOKING", 
            "COOKING": "PACKING", 
            "PACKING": "LOADING",
            "LOADING": "DELIVERY",
            "DELIVERY": "DELIVERED"
        }
        return next_states.get(current_state)

    def assign_resources(self, batch, target_state):
        assigned_station = None
        assigned_driver = None
        
        if target_state in ["PREPARING", "COOKING", "PACKING"]:
            for s_id, s_info in self.stations.items():
                if s_info["role"] == target_state and s_info["state"] == "IDLE":
                    assigned_station = s_id
                    break
                    
        elif target_state == "LOADING":
            for d_id, status in self.drivers_status.items():
                if status["state"] == "LOADING" and status["canteen_id"] == batch.canteen_id:
                    assigned_driver = d_id
                    break
            if not assigned_driver:
                for d_id, status in self.drivers_status.items():
                    if status["state"] == "IDLE":
                        assigned_driver = d_id
                        break
                        
        return assigned_station, assigned_driver

    def update_resource_states(self, batch, target_state, assigned_station, assigned_driver):
        if batch.current_station_id and target_state in ["COOKING", "PACKING", "LOADING"]:
            self.stations[batch.current_station_id]["state"] = "IDLE"
            self.stations[batch.current_station_id]["batch_id"] = None
            
        if assigned_station:
            self.stations[assigned_station]["state"] = "BUSY"
            self.stations[assigned_station]["batch_id"] = batch.batch_id
            batch.current_station_id = assigned_station
            
        elif assigned_driver:
            batch.driver_id = assigned_driver
            self.drivers_status[assigned_driver]["state"] = "LOADING"
            self.drivers_status[assigned_driver]["canteen_id"] = batch.canteen_id
            self.drivers_status[assigned_driver]["active_batches"] += 1
            batch.current_station_id = None

    def build_payload(self, batch, b_id):
        current_state = batch.state
        ts = self.global_clock.isoformat(timespec='minutes')
        
        if current_state == "DELIVERY":
            self.drivers_status[batch.driver_id]["state"] = "DELIVERING"
        
        if current_state in ["PREPARING", "COOKING", "PACKING"]:
            topic = "kitchen_station_events"
            event_type = "kitchen_action"
            self.kitchen_event_counter += 1
            evt_id = f"KIT-EVT-{self.kitchen_event_counter}"
            
            if random.random() < 0.03:
                batch.current_weight = round(batch.current_weight - random.uniform(2, 5), 2)
            
            weight = batch.current_weight
            if random.random() < 0.05:
                weight = None

            if current_state == "PREPARING":
                temp = round(random.uniform(15.0, 22.0), 1) 
            elif current_state == "PACKING":
                temp = round(random.uniform(40.0, 60.0), 1) 
            else: 
                if random.random() < 0.10:
                    temp = round(random.uniform(50.0, 74.0), 1) 
                else:
                    temp = round(random.uniform(75.0, 100.0), 1) 
                
            if random.random() < 0.05:
                temp = None

            payload_data = {
                "batch_id": b_id,
                "station_id": batch.current_station_id,
                "recipe_id": batch.recipe,
                "action": current_state,
                "weight_kg": weight,
                "ingredients": self.get_ingredients(batch.recipe, batch.initial_weight),
                "temperature_celsius": temp,
                "event_timestamp": ts
            }
        else:
            topic = "dispatch_events"
            event_type = "dispatch_action"
            self.dispatch_event_counter += 1
            evt_id = f"DIS-EVT-{self.dispatch_event_counter}"
            
            if batch.truck_temp is not None:
                truck_temp = round(batch.truck_temp + random.uniform(-0.5, 0.5), 1)
            else:
                truck_temp = 4.0 
                
            if random.random() < 0.05:
                truck_temp = None
                
            payload_data = {
                "batch_id": b_id,
                "canteen_id": batch.canteen_id,
                "driver_id": batch.driver_id,
                "action": current_state,
                "truck_temp_celsius": truck_temp,
                "event_timestamp": ts
            }
            
        return topic, evt_id, event_type, payload_data

    def build_envelope(self, evt_id, event_type, payload_data, source):
        envelope = (
            {
                "metadata": {
                    "event_id": evt_id,
                    "event_type": event_type,
                    "source": source,
                    "generated_at": datetime.now().isoformat()
                },
                "payload": payload_data
            }
        )

        if random.random() < 0.05:
            fail_type = random.choice(["not_dict", "missing_meta", "missing_payload", "missing_id", "missing_type", "missing_action"])
            if fail_type == "not_dict":
                return "CORRUPT_RAW_STRING_DATA"
            elif fail_type == "missing_meta":
                del envelope["metadata"]
            elif fail_type == "missing_payload":
                del envelope["payload"]
            elif fail_type == "missing_id":
                if random.random() < 0.5:
                    del envelope["metadata"]["event_id"]
                else:
                    del envelope["payload"]["batch_id"]
            elif fail_type == "missing_type":
                del envelope["metadata"]["event_type"]
            elif fail_type == "missing_action":
                del envelope["payload"]["action"]

        return envelope
        
    def handle_delivery(self, batch, b_id):
        d_id = batch.driver_id
        self.drivers_status[d_id]["active_batches"] -= 1
        
        if self.drivers_status[d_id]["active_batches"] == 0:
            self.drivers_status[d_id]["state"] = "RETURNING"
            self.drivers_status[d_id]["return_time"] = self.global_clock + timedelta(minutes=batch.trip_duration)
            
        del self.active_batches[b_id]

    def generate_event(self):
        self.update_clock()
        
        if self.check_day_limits() == "DONE":
            return "DONE", None
            
        self.spawn_new_batches()
        
        events_to_send = []
        batch_ids = sorted(self.active_batches.keys())
        
        for b_id in batch_ids:
            batch = self.active_batches[b_id]
            
            if not batch.can_advance(self.global_clock):
                continue
                
            target_state = self.get_target_state(batch.state)
            assigned_station, assigned_driver = self.assign_resources(batch, target_state)
            
            if target_state in ["PREPARING", "COOKING", "PACKING"] and not assigned_station:
                continue
            if target_state == "LOADING" and not assigned_driver:
                continue
                
            self.update_resource_states(batch, target_state, assigned_station, assigned_driver)
            
            mins_for_next_task = batch.advance()
            batch.ready_time = self.global_clock + timedelta(minutes=mins_for_next_task)
            
            topic, evt_id, event_type, payload_data = self.build_payload(batch, b_id)
            source = self.source
            envelope = self.build_envelope(evt_id, event_type, payload_data, source)
            
            if batch.state == "DELIVERED":
                self.handle_delivery(batch, b_id)
                
            self.total_rows_made += 1
            events_to_send.append((topic, envelope))

        if events_to_send:
            return "MULTI", events_to_send
        return None, None

import json
from kafka import KafkaProducer
from kafka.errors import NoBrokersAvailable

class KafkaPublisher:
    def __init__(self, config_file='config.json'):
        self.config = load_config(config_file)
        self.producer = self.connectToKafka()

    def connectToKafka(self):
        servers = self.config.get("kafka_bootstrap_servers", "localhost:9092")
        try:
            producer = KafkaProducer(
                bootstrap_servers=servers,
                key_serializer=lambda k: k.encode('utf-8') if k else None,
                value_serializer=lambda v: json.dumps(v).encode('utf-8'),
                acks='all',
                retries=3
            )
            print("Successfully connected to Kafka.")
            return producer
        except NoBrokersAvailable:
            print(f"Failed to connect to Kafka at {servers}. Check your network.")
            return None
            
    def on_send_error(self, excp):
        print(f"Failed to send message to Kafka: {excp}")

    def run(self):
        if not self.producer:
            print("Cannot start generator without a Kafka connection.")
            return

        gen = EventGenerator(self.config)
        print("Generator running... Press Ctrl+C to stop.")

        try:
            while True:
                status, events = gen.generate_event()

                if status == "DONE":
                    print("Limits reached. All batches delivered. Stopping generator.")
                    break

                if status == "MULTI":
                    for topic, data in events:
                        message_key = None
                        if type(data) is dict and "metadata" in data:
                            message_key = data["metadata"].get("event_id")
                        topic_name = self.config.get(topic, topic)
                        future = self.producer.send(topic_name, key=message_key, value=data)
                        future.add_errback(self.on_send_error)
                        
                #time.sleep(0.5)
        
        except KeyboardInterrupt:
            print("\nManual stop received.")
        except Exception as e:
            print(f"An unexpected error happened: {e}")
        finally:
            if self.producer:
                print("Closing Kafka connection...")
                self.producer.flush()
                self.producer.close()

"""import json
from tools.utils import load_config
from simulation.generator import EventGenerator

class BackupSaver:
    def __init__(self, config_file='config.json', backup_file='backup_data.jsonl'):
        self.config = load_config(config_file)
        self.backup_file = backup_file
        self.generator = EventGenerator(self.config)

    def _format_record(self, topic, data):
        record = {
            "target_topic": topic,
            "message": data
        }
        return json.dumps(record)

    def save(self):
        print(f"Generating data and saving to {self.backup_file}...")
        print("Press Ctrl+C to stop.")

        try:
            with open(self.backup_file, 'w') as file:
                while True:
                    status, events = self.generator.generate_event()

                    if status == "DONE":
                        print("Simulation finished.")
                        break

                    if status == "MULTI":
                        for topic, data in events:
                            json_string = self._format_record(topic, data)
                            file.write(json_string + "\n")

        except KeyboardInterrupt:
            print("\nStopped saving data.")

if __name__ == "__main__":
    app = BackupSaver('config.json', 'backup_data.jsonl')
    app.save()"""

"""import json
import time
import os
from kafka import KafkaProducer
from tools.utils import load_config

class BackupReplayer:
    def __init__(self, config_file='config.json', backup_file='backup_data.jsonl'):
        self.config = load_config(config_file)
        self.backup_file = backup_file
        servers = self.config.get("kafka_bootstrap_servers", "localhost:9092")
        self.producer = None

        try:
            self.producer = KafkaProducer(
                bootstrap_servers=servers,
                value_serializer=lambda v: json.dumps(v).encode('utf-8')
            )
        except Exception as e:
            print(f"Failed to start producer: {e}")

    def _is_valid_file(self):
        if not os.path.exists(self.backup_file):
            print(f"Error: The file {self.backup_file} does not exist.")
            return False
            
        if not self.backup_file.endswith('.jsonl'):
            print(f"Error: The file {self.backup_file} must end with .jsonl.")
            return False
            
        return True

    def replay(self):
        if not self.producer or not self._is_valid_file():
            return

        print(f"Reading from {self.backup_file} and sending to Kafka...")

        try:
            with open(self.backup_file, 'r') as file:
                for line in file:
                    try:
                        record = json.loads(line)
                        
                        # We extract the routing info. The consumer handles the rest.
                        topic_key = record.get("target_topic")
                        data = record.get("message")
                        
                        if topic_key and data:
                            real_topic = self.config.get(topic_key, topic_key)
                            self.producer.send(real_topic, data)
                            print(f"Sent event to {real_topic}")
                            
                            time.sleep(0.5)
                            
                    except json.JSONDecodeError:
                        print("Warning: Skipped an unreadable text line.")
                        
        except KeyboardInterrupt:
            print("\nManual stop received. Halting file read.")
        except Exception as e:
            print(f"An unexpected error happened: {e}")
        finally:
            if self.producer:
                self.producer.flush()
                self.producer.close()
            print("Finished sending backup data.")"""

if __name__ == "__main__":
    KafkaPublisher('config.json').run()
    #backupapp = BackupSaver('config.json', 'backup_data.jsonl')
    #backupapp.save()
    #replayapp = BackupReplayer('config.json', 'backup_data.jsonl')
    #replayapp.replay()

Successfully connected to Kafka.
Generator running... Press Ctrl+C to stop.
Limits reached. All batches delivered. Stopping generator.
Closing Kafka connection...
